In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import re

In [2]:
raw_path = Path.cwd().resolve().parent / 'data'/ 'raw'

In [3]:
snapshot_folder = raw_path / "2026-06-23"

calendar_path = snapshot_folder / "calendar.csv.gz"
listings_path = snapshot_folder / "listings.csv.gz"
reviews_path = snapshot_folder / "reviews.csv.gz"

calendar = pd.read_csv(calendar_path, encoding='utf-8', dtype={"listing_id": str})

listings = pd.read_csv(listings_path, encoding='utf-8', dtype={"id": str, "host_id": str, "listing_id": str})

reviews = pd.read_csv(reviews_path, encoding='utf-8', dtype={"id": str, "listing_id": str, "reviewer_id": str, "review_id": str})

In [4]:
import sys
import yaml
from pathlib import Path

project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from src import kpi 


In [5]:
from pathlib import Path
config_path = Path.cwd().parent / "configs" / "porto.yaml"
kpi.calculate_review_concentration(listings, reviews, output_path = "data/processed/kpi_review_concentration.csv")

Đã lưu KPI Tỉ lệ tập trung đánh giá tại: data/processed/kpi_review_concentration.csv


(     year analysis_area  area_reviews  city_total_reviews  concentration_pct
 0    2012           A02             2                   3          66.666667
 1    2012           A08             1                   3          33.333333
 2    2013           A01             7                  25          28.000000
 3    2013           A02            11                  25          44.000000
 4    2013           A06             3                  25          12.000000
 ..    ...           ...           ...                 ...                ...
 97   2025           A04            53                 687           7.714702
 98   2025           A05            64                 687           9.315866
 99   2025           A06            40                 687           5.822416
 100  2025           A07            91                 687          13.245997
 101  2025           A08            60                 687           8.733624
 
 [102 rows x 5 columns],
 analysis_area    A01    A02    A03  

In [ ]:
kpi.calculate_monthly_occupancy_kpi(calendar, listings, output_path = "data/processed/kpi_occupancy_rate.csv")

In [ ]:
from pathlib import Path
config_path = Path.cwd().parent / "configs" / "porto.yaml"
kpi.calculate_dormant_listing_rate_over_time(config_path, output_path = "data/processed/kpi_dormant_listing_rate.csv")

In [ ]:
kpi.calculate_value_score_by_price_tier_single_snapshot(listings,output_path = "data/processed/kpi_value_score_by_price_tier.csv")

In [ ]:
kpi.calculate_overpriced_rate(
    listings,
    premium_ratio=1.25,
    low_value_threshold=4.6,
    output_path="data/processed/kpi_overpriced_rate.csv",
)

In [ ]:
kpi.calculate_review_velocity_over_time(
    config_path,
    delta_months=3,
    output_path="data/processed/kpi_review_velocity.csv",
)

In [ ]:
kpi.calculate_revpar_single_snapshot(
    listings,
    output_path="data/processed/kpi_revpar.csv"
)

In [6]:
kpi.calculate_professional_host_share_over_time(
    config_path,
    pro_threshold=10,
    latest_only=False,
    output_path="data/processed/kpi_professional_host_share.csv",
)

,snapshot,analysis_area,individual_listings,small_multi_listings,professional_listings,total_listings,individual_share_%,small_multi_share_%,professional_share_%
0,2025-09-21,A01,35,54,38,127,27.56,42.52,29.92
1,2025-09-21,A02,21,55,53,129,16.28,42.64,41.09
2,2025-09-21,A03,17,53,50,120,14.17,44.17,41.67
3,2025-09-21,A04,34,48,38,120,28.33,40.00,31.67
4,2025-09-21,A05,23,57,45,125,18.40,45.60,36.00
5,2025-09-21,A06,44,60,22,126,34.92,47.62,17.46
6,2025-09-21,A07,27,48,55,130,20.77,36.92,42.31
7,2025-09-21,A08,27,46,50,123,21.95,37.40,40.65
8,2025-12-25,A01,42,56,29,127,33.07,44.09,22.83
9,2025-12-25,A02,26,59,44,129,20.16,45.74,34.11


In [ ]:
# Thông tin file Listings

# print((listings.isna().mean().sort_values(ascending=False).head(15))*100)
# print("\n")
# listings.info()

In [ ]:
# # Các thống số file Listings
# print(listings["room_type"].value_counts())
# print("Khong co review: ", listings["number_of_reviews"].eq(0).sum())
# print("Thieu price: ", listings["price"].isna().sum())
# listings.groupby("analysis_area")["id"].size().sort_values(ascending=False)

In [ ]:
print(len(calendar))
print(len(reviews))
print(reviews["Mẫu LLM"].notna().sum())
print(reviews["Nhãn tay"].notna().sum())

In [ ]:
rv_to_label = reviews[reviews["Mẫu LLM"].str.strip().str.lower()=='x'][["id", "comments"]]
samples_path = Path.cwd().resolve().parent / 'data'/ 'samples'
samples_path.mkdir(parents=True, exist_ok=True)
save_path = samples_path / 'rv_llm_baseline.csv'
# rv_to_label.to_csv(save_path, index=False, encoding='utf-8')


In [ ]:
d = pd.read_csv(save_path, encoding='utf-8')
d.head(10)["comments"].loc[1]

In [ ]:
PAT_SENTIMENT_GOOD = r"amazing|excellent|wonderful|fantastic|perfect|great|lovely|awesome|beautiful|superb|outstanding|fabulous|incredible|brilliant|exceptional|enjoyed|loved|pleasant|delightful|recommend|happy|satisfied|would stay again|had a great time|loved our stay|exceeded expectations|everything was perfect"
PAT_ACCESS_GOOD = r"central|city center|city centre|well located|convenient|accessible|walkable|walking distance|close to|near|nearby|easy access|public transport|metro nearby|heart of|short walk|minutes walk|easy to reach|well connected"
PAT_QUIET = r"quiet|very quiet|peaceful|silent|tranquil|calm|soundproof|no noise|no disturbance|not noisy|quiet neighborhood|peaceful nights|no street noise|well insulated|couldn't hear anything"

PAT_SENTIMENT_BAD = r"terrible|awful|horrible|bad|poor|worst|disappointing|disappointed|unpleasant|unacceptable|frustrating|uncomfortable|dirty|regret|problem|issues|complaint|avoid|rude|would not recommend|never again|not worth it|wouldn't stay again|did not enjoy"
PAT_ACCESS_BAD = r"far from|far away|remote|isolated|inconvenient|difficult to reach|hard to reach|long walk|too far|not central|not accessible|difficult access|far from metro|far from center"
PAT_NOISY = r"noisy|very noisy|too noisy|loud|very loud|traffic noise|street noise|road noise|construction noise|loud music|disturbed sleep|noise at night|loud neighbors|noise problem|unbearable noise"

PAT_SENTIMENT_NEUTRAL = r"okay|ok|average|fine|fair|acceptable|normal|ordinary|nothing special|so-so|could be better|as expected"
PAT_ACCESS_NEUTRAL = r"location was okay|location was fine|reasonable location|location was acceptable|not too far|not too close|moderate distance|location was alright"
PAT_NOISE_NEUTRAL = r"some noise|a little noisy|slightly noisy|bit noisy|minor noise|not too noisy|manageable noise|tolerable noise|noise was okay|noise was fine|noise was acceptable|didn't mind the noise"

def baseline_test(rv_to_label): 
    rv = rv_to_label.copy()
    sentiment_good = rv["comments"].str.findall(PAT_SENTIMENT_GOOD, flags=re.I)
    nums_of_good_sentiment = sentiment_good.map(lambda x: len(x))

    sentiment_bad = rv["comments"].str.findall(PAT_SENTIMENT_BAD, flags=re.I)
    nums_of_bad_sentiment = sentiment_bad.map(lambda x: len(x))
    
    sentiment_neutral = rv["comments"].str.findall(PAT_SENTIMENT_NEUTRAL, flags=re.I)
    nums_of_neutral_sentiment = sentiment_neutral.map(lambda x: len(x))
    
    access_good = rv["comments"].str.contains(PAT_ACCESS_GOOD, flags=re.I)
    access_bad = rv["comments"].str.contains(PAT_ACCESS_BAD, flags=re.I)
    access_neutral = rv["comments"].str.contains(PAT_ACCESS_NEUTRAL, flags=re.I)
    
    noise_quiet = rv["comments"].str.contains(PAT_QUIET, flags=re.I)
    noise_noisy = rv["comments"].str.contains(PAT_NOISY, flags=re.I)
    noise_neutral = rv["comments"].str.contains(PAT_NOISE_NEUTRAL, flags=re.I)

    sentiment_score = nums_of_good_sentiment - nums_of_bad_sentiment
    rv["baseline_general_sentiment"] = ""
    rv.loc[sentiment_score > 0, "baseline_general_sentiment"] = "Positive"
    rv.loc[sentiment_score < 0, "baseline_general_sentiment"] = "Negitive"
    rv.loc[sentiment_score == 0, "baseline_general_sentiment"] = "Mixed" 

    rv["baseline_accessibility"] = "Not mentioned"
    rv.loc[access_good, "baseline_accessibility"] = "Highly accessible"
    rv.loc[access_bad, "baseline_accessibility"] = "Poor access"
    rv.loc[access_neutral, "baseline_accessibility"] = "Neutral"

    rv["baseline_noise"] = "Not mentioned"
    rv.loc[noise_quiet, "baseline_noise"] = "Quiet"
    rv.loc[noise_noisy, "baseline_noise"] = "Noisy"
    rv.loc[noise_neutral, "baseline_noise"] = "Acceptable"

    return rv

In [ ]:
import sys
import yaml
from pathlib import Path

project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from src import llm

config_path = Path.cwd().resolve().parent / "configs" / "porto.yaml"
with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)

In [ ]:
nhan_tay = pd.read_csv(samples_path / "rv_to_label.csv", encoding='utf-8')
labeled = llm.llm(nhan_tay, config_path)

In [ ]:
llm = labeled.add_prefix("llm_")
llm = llm.rename(columns={"llm_comments": "comments"})
so_sanh = nhan_tay.merge(llm, how='left', on="comments")
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].sum()

In [ ]:
so_sanh[so_sanh["accessibility"] != so_sanh["llm_accessibility"]]

In [ ]:
so_sanh[so_sanh["accessibility"] != so_sanh["llm_accessibility"]].loc[14, "comments"]

In [ ]:
so_sanh[so_sanh["noise"] != so_sanh["llm_noise"]]

In [ ]:
so_sanh[so_sanh["noise"] != so_sanh["llm_noise"]].loc[97, "comments"]

In [ ]:
so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]]

In [ ]:
 so_sanh[so_sanh["general_sentiment"] != so_sanh["llm_general_sentiment"]].loc[0,"comments"]